# 01e Synthetic-To-Real Transfer

This notebook runs Experiment 5 for the journal article: M8 is trained on synthetic data and evaluated on actual manually labelled test-period data, with M7 as a deterministic baseline.

The scientific question is whether synthetic wrong-positive generation teaches the model useful patterns for real manually labelled wrong-positive readings. This is the bridge between the controlled synthetic benchmark and operational actual-data usefulness.

Smoke mode validates the transfer setup without fitting the synthetic-trained M8 model.

## Imports And Path Setup

This section locates the journal article folder and imports the shared experiment helpers.

The notebook needs both synthetic and actual datasets, so reliable root detection matters. The helper lookup searches upward from the current working directory until it finds the journal article structure.

If this cell fails, confirm the notebook is being run from inside the PyNRPF repository and that `_experiment_helpers.py` exists.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
helper_dir = None
for candidate in [start, *start.parents]:
    direct = candidate if candidate.name == "notebooks" else candidate / "publication" / "2_journal_article" / "notebooks"
    if (direct / "_experiment_helpers.py").exists():
        helper_dir = direct
        break
if helper_dir is None:
    raise RuntimeError("Could not locate _experiment_helpers.py")
sys.path.insert(0, str(helper_dir))

from _experiment_helpers import (
    dataset_summary,
    experiment_output_dir,
    find_article_root,
    load_config,
    load_dataset,
    run_transfer_experiment,
    time_masks,
)

ARTICLE_ROOT = find_article_root(start)
NOTEBOOK_NAME = "01e_synthetic_to_real_transfer.ipynb"
EXPERIMENT_ID = "experiment_5_synthetic_to_real_transfer"
ARTICLE_ROOT

## Load YAML Config

Editable constants live in `config/experiment_config.yaml` rather than being hard-coded in this notebook.

Important values loaded from YAML include dataset paths, split dates, active methods, M8 hyperparameters and thresholds, M7 threshold settings, resume behavior, evaluation rules, and output folder names.

`methods.enabled` is the only place to choose methods. Use `["m7_dtr"]` to run M7 only, or `["m8_xgb", "m7_dtr"]` to run both. When `m8_xgb` is absent, the helper does not train M8 at all.

The most important safety flag is `execution.run_full_experiment`. When it is `false`, the notebook performs smoke validation only. When it is `true`, it can train models and write full CSV outputs. Resume behavior is controlled by `resume.skip_completed`; completed fold/method tasks are skipped unless `execution.overwrite_outputs` is set to `true`.


In [ ]:
cfg = load_config(ARTICLE_ROOT)
active_methods = cfg["methods"]["enabled"]
print("Config:", cfg["_config_path"])
print("Run full experiment:", cfg["execution"]["run_full_experiment"])
print("Enabled methods:", active_methods)
print("M8 enabled:", "m8_xgb" in active_methods)
print("M8 behavior:", "train only for incomplete M8 tasks" if "m8_xgb" in active_methods else "skip all M8 training")
print("Resume skip completed:", cfg.get("resume", {}).get("skip_completed", False))
print("Overwrite outputs:", cfg["execution"].get("overwrite_outputs", False))
print("Output folder:", experiment_output_dir(ARTICLE_ROOT, cfg, EXPERIMENT_ID))


## Load And Validate Datasets

This reads both processed CSV datasets and confirms they share the canonical schema.

The synthetic dataset supplies M8 training rows. The actual dataset supplies the held-out transfer evaluation rows. Both must use the same column names and label meanings for the transfer experiment to be valid.

The printed summaries make it easier to catch accidental use of stale datasets or mismatched date ranges.

In [ ]:
synthetic = load_dataset(ARTICLE_ROOT, cfg, "synthetic")
actual = load_dataset(ARTICLE_ROOT, cfg, "actual")
print(dataset_summary(synthetic, cfg, "synthetic"))
print(dataset_summary(actual, cfg, "actual"))

## Define Transfer Split

M8 trains on the synthetic train period and evaluates on the actual test period.

Synthetic training rows use `2021-11-01` through `2023-09-30`. Actual transfer test rows use `2023-10-01` through `2024-09-30`.

This keeps the transfer experiment aligned with the conference time split while changing the data domain from synthetic to actual.

In [ ]:
synthetic_train_mask, _ = time_masks(synthetic, cfg)
_, actual_test_mask = time_masks(actual, cfg)
print("Synthetic train rows:", int(synthetic_train_mask.sum()))
print("Actual test rows:", int(actual_test_mask.sum()))

## Method Helpers

M8 and M7 execution is implemented in the shared helper module to keep this notebook readable while keeping all experiment notebooks consistent.

Active methods come from `methods.enabled` in YAML. The notebook does not maintain a second method list. If the list is only `["m7_dtr"]`, M7 runs independently and no M8 feature building or XGBoost training is attempted. If `m8_xgb` is enabled, M8 training happens only for fold/method tasks that are missing a complete checkpoint.

M8 is the trainable two-stage XGBoost method: first a day-level classifier, then an interval-level classifier inside candidate days. M7 is the deterministic threshold-rule baseline and does not train.

Both methods use fixed conference-paper settings from YAML. This is deliberate: the journal experiments compare generalization settings, not retuned parameter sets.


In [ ]:
active_methods = cfg["methods"]["enabled"]
print("Enabled methods:", active_methods)
print("M8 enabled:", "m8_xgb" in active_methods)
print("M8 behavior:", "train only for incomplete M8 tasks" if "m8_xgb" in active_methods else "skip all M8 training")
print("M7 enabled:", "m7_dtr" in active_methods)
print("Resume skip completed:", cfg.get("resume", {}).get("skip_completed", False))
print("Overwrite outputs:", cfg["execution"].get("overwrite_outputs", False))
print("M8 thresholds:", cfg["m8_xgb"]["xgb1_day"]["threshold"], cfg["m8_xgb"]["xgb2_timestamp"]["threshold"])


## Run Experiment

Smoke mode writes a manifest only and skips synthetic M8 training.

Full mode trains M8 on synthetic train-period rows, evaluates M8 and M7 on actual test-period rows, then writes metrics and prediction CSV files.

Because this experiment crosses domains, review the manifest carefully before interpreting the metrics.

In full mode, each fold/method task writes three checkpoint files as soon as that task completes: a prediction CSV under `predictions/`, a per-task metrics CSV under `metrics/`, and a completion YAML under `status/`. The completion marker is written last, so interrupted or failed tasks rerun cleanly.

On rerun, completed tasks are skipped when `resume.skip_completed: true` and `execution.overwrite_outputs: false`. The notebook also rebuilds `metrics_summary.csv`, `fold_metrics.csv`, and `manifest.yaml` from completed task files each time, so partial output remains inspectable even if a long run stops halfway.


In [ ]:
run_transfer_experiment(ARTICLE_ROOT, cfg, EXPERIMENT_ID, NOTEBOOK_NAME)

## Quick Review

Use this final cell to confirm where the notebook wrote outputs.

After a smoke run, expect `manifest.yaml` plus created `predictions/`, `metrics/`, and `status/` folders. After a full or partial run, inspect `status/` first to see which fold/method tasks completed, then inspect the per-task metrics and prediction CSVs.

`metrics_summary.csv`, `fold_metrics.csv`, and `manifest.yaml` are rebuilt from completed task files on every run. If results look wrong later, start debugging by checking the status YAML for the affected fold/method, then compare the manifest settings against `experiment_config.yaml`.


In [ ]:
print("Review output folder:", experiment_output_dir(ARTICLE_ROOT, cfg, EXPERIMENT_ID))